# AAPL/CAT Portfolio: Parametric vs Historical vs Monte Carlo VaR/ES

**Purpose:** Build the AAPL + CAT equity portfolio from the course homework, compute 5-day 99% VaR and 97.5% ES under four methods, and compare results. Purchase date: 10/13/1997. AAPL last on that date: 0.2026 (split-adjusted).

In [ ]:
%matplotlib inline
import sys
sys.path.insert(0, "..")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import date

PURCHASE_DATE  = '10/13/1997'
SHARES_AAPL    = 24679
SHARES_CAT     = 171
HORIZON_DAYS   = 5
VAR_CONF       = 0.99
ES_CONF        = 0.975
LOOKBACK       = 2 * 252  # 2-year window
print('Parameters set.')

## Section 1 — Load and Align Data

In [ ]:
aapl = pd.read_csv('../data/AAPL-bloomberg.csv', parse_dates=['Dates'], dayfirst=False)
aapl = aapl.set_index('Dates').sort_index()['PX_LAST']

cat = pd.read_csv('../data/CAT-bloomberg.csv', parse_dates=['Dates'], dayfirst=False)
cat = cat.set_index('Dates').sort_index()['PX_LAST']

prices_all = pd.DataFrame({'AAPL': aapl, 'CAT': cat}).dropna()
print(f'Aligned rows: {len(prices_all)}, '
      f'{prices_all.index[0].date()} to {prices_all.index[-1].date()}')

## Section 2 — Portfolio Construction from Purchase Date

In [ ]:
start = pd.Timestamp(PURCHASE_DATE)
prices = prices_all[prices_all.index >= start].copy()
portfolio_value_ts = SHARES_AAPL * prices['AAPL'] + SHARES_CAT * prices['CAT']
latest_pv = portfolio_value_ts.iloc[-1]
print(f'Portfolio value on {prices.index[-1].date()}: ${latest_pv:,.2f}')
print(f'(Expected: ~$6,931,589 from homework)')
print(f'AAPL last price: ${prices["AAPL"].iloc[-1]:.4f}')
print(f'CAT  last price: ${prices["CAT"].iloc[-1]:.4f}')

## Section 3 — Log Returns and Summary Statistics

In [ ]:
log_ret = np.log(prices / prices.shift(1)).dropna()

# 2-year lookback window
window = log_ret.tail(LOOKBACK)
mu_ann   = window.mean() * 252
sig_ann  = window.std()  * np.sqrt(252)
corr     = window.corr()

print('Annualized mu (2y lookback):')
print(mu_ann.to_string())
print()
print('Annualized sigma (2y lookback):')
print(sig_ann.to_string())
print()
print('Correlation matrix:')
print(corr.to_string())

## Section 4 — Exact GBM VaR/ES (portfolio-level lognormal)

In [ ]:
from src.risk.lognormal import var_long_lognormal, es_long_lognormal

# Portfolio log-returns over the 2-year lookback
port_log_ret = np.log(portfolio_value_ts / portfolio_value_ts.shift(1)).dropna()
port_window  = port_log_ret.tail(LOOKBACK)

# Annualized parameters (GBM parameterisation: mu = drift of log-return + 0.5*sigma^2)
mu_port_daily    = port_window.mean()
sig_port_daily   = port_window.std()
mu_port_ann      = mu_port_daily * 252 + 0.5 * (sig_port_daily * np.sqrt(252))**2
sig_port_ann     = sig_port_daily * np.sqrt(252)

V0_port = portfolio_value_ts.iloc[-1]

gbm_var = var_long_lognormal(V0_port, mu_port_ann, sig_port_ann, HORIZON_DAYS/252, VAR_CONF)
gbm_es  = es_long_lognormal( V0_port, mu_port_ann, sig_port_ann, HORIZON_DAYS/252, ES_CONF)
print(f'GBM Exact VaR 99%  (5-day): ${gbm_var:,.2f}  (expected ≈ 579,445)')
print(f'GBM Exact ES  97.5% (5-day): ${gbm_es:,.2f}  (expected ≈ 662,028)')

## Section 5 — Two-Stock Normal Approximation

In [ ]:
from src.risk.normal import portfolio_delta_normal_mean_var, normal_var, normal_es

# Last prices
S_aapl = prices['AAPL'].iloc[-1]
S_cat  = prices['CAT'].iloc[-1]
exposures = np.array([SHARES_AAPL * S_aapl, SHARES_CAT * S_cat])

# Horizon mean and covariance (scale from daily to h-day)
mu_daily  = window.mean().values          # shape (2,)
cov_daily = window.cov().values           # shape (2,2)
mu_h  = mu_daily  * HORIZON_DAYS
cov_h = cov_daily * HORIZON_DAYS

port_mean, port_std = portfolio_delta_normal_mean_var(exposures, mu_h, cov_h)
two_stock_var = normal_var(port_mean, port_std, VAR_CONF)
two_stock_es  = normal_es(port_mean, port_std, ES_CONF)
print(f'Two-Stock Normal VaR 99%  (5-day): ${two_stock_var:,.2f}  (expected ≈ 597,001)')
print(f'Two-Stock Normal ES  97.5% (5-day): ${two_stock_es:,.2f}  (expected ≈ 600,071)')

## Section 6 — Historical Simulation VaR/ES

In [ ]:
from src.risk.historical import historical_var_es
from src.schemas import Portfolio, StockPosition

portfolio_obj = Portfolio(
    stocks=[
        StockPosition(ticker='AAPL', quantity=SHARES_AAPL),
        StockPosition(ticker='CAT',  quantity=SHARES_CAT),
    ]
)

pricing_dt = prices.index[-1].date()
hist_result = historical_var_es(
    portfolio=portfolio_obj,
    prices=prices,
    pricing_date=pricing_dt,
    lookback_days=LOOKBACK,
    horizon_days=HORIZON_DAYS,
    var_confidence=VAR_CONF,
    es_confidence=ES_CONF,
    shock_type='log',
)
print(f'Historical VaR 99%  (5-day): ${hist_result["var"]:,.2f}')
print(f'Historical ES  97.5% (5-day): ${hist_result["es"]:,.2f}')
print(f'Scenarios used: {hist_result["n_scenarios"]}')

## Section 7 — Monte Carlo VaR/ES

In [ ]:
from src.risk.monte_carlo import monte_carlo_var_es

mc_result = monte_carlo_var_es(
    portfolio=portfolio_obj,
    prices=prices,
    pricing_date=pricing_dt,
    lookback_days=LOOKBACK,
    horizon_days=HORIZON_DAYS,
    var_confidence=VAR_CONF,
    es_confidence=ES_CONF,
    n_simulations=5000,
    random_seed=42,
)
print(f'Monte Carlo VaR 99%  (5-day): ${mc_result["var"]:,.2f}')
print(f'Monte Carlo ES  97.5% (5-day): ${mc_result["es"]:,.2f}')
print(f'Simulations: {mc_result["n_simulations"]}')

## Section 8 — Method Comparison Table

In [ ]:
print(f'{'Method':<30} {'VaR 99% ($)':>15} {'ES 97.5% ($)':>15}')
print('-' * 62)
rows = [
    ('GBM Exact (Lognormal)',     gbm_var,          gbm_es),
    ('Two-Stock Normal',           two_stock_var,    two_stock_es),
    ('Historical Simulation',      hist_result['var'], hist_result['es']),
    ('Monte Carlo (N=5000)',        mc_result['var'],   mc_result['es']),
]
for name, v, e in rows:
    print(f'{name:<30} {v:>15,.0f} {e:>15,.0f}')

## Section 9 — Portfolio Value Time Series + Rolling VaR

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=False)

# Plot 1: Portfolio value
ax1.plot(portfolio_value_ts.index, portfolio_value_ts.values / 1e6, color='steelblue')
ax1.set_ylabel('Portfolio Value ($M)')
ax1.set_title('AAPL+CAT Portfolio Value (Since Purchase 1997)')
ax1.grid(True, alpha=0.3)

# Plot 2: Rolling 2-year 5-day 99% GBM VaR at annual intervals
years = pd.date_range(start='2002-01-01', end=prices.index[-1], freq='YS')
var_dates, var_vals = [], []
for yr in years:
    subset = port_log_ret[port_log_ret.index <= yr].tail(LOOKBACK)
    if len(subset) < 100:
        continue
    mu_d  = subset.mean()
    sig_d = subset.std()
    V_yr  = portfolio_value_ts.loc[portfolio_value_ts.index <= yr].iloc[-1]
    mu_a  = mu_d * 252 + 0.5 * (sig_d * np.sqrt(252))**2
    sig_a = sig_d * np.sqrt(252)
    var_yr = var_long_lognormal(V_yr, mu_a, sig_a, HORIZON_DAYS/252, VAR_CONF)
    var_dates.append(yr)
    var_vals.append(var_yr / 1e6)

ax2.bar(var_dates, var_vals, width=250, color='firebrick', alpha=0.7, label='GBM VaR 99% (annual, $M)')
ax2.set_ylabel('5-day 99% VaR ($M)')
ax2.set_title('Rolling 2-Year 5-Day 99% VaR at Annual Intervals')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()